# Corrección compatible del notebook original
Se corrigió el retorno del checkpoint a (ruta, época), incluido el caso sin checkpoints. Se conserva el procedimiento de entrenamiento original. Esta copia no resuelve por sí sola la calidad del dataset ni reanuda el optimizador. Ejecuta las celdas previas de datos antes de entrenar.


# Fine-tuning del modelo de embeddings — SW_DERECHO / dataset TSJ (Google Colab)

Afina un modelo de `sentence-transformers` con los pares hecho → artículo
sacados de `genesis.tsj.bo`, para mejorar el componente semántico del
`RankingService` (60% del score del ranking).

**Antes de correr:** arriba a la izquierda, *Runtime → Change runtime type →
T4 GPU*. Después subí tu `dataset_completo.xlsx` a tu Google Drive (a la
carpeta que quieras) — la celda de abajo monta tu Drive para leerlo de ahí.

## 1. Dependencias
Kaggle ya trae PyTorch instalado; solo falta `sentence-transformers`.

In [ ]:
!pip install -q -U sentence-transformers

import torch
print("GPU disponible:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")

## 2. Cargar el dataset
`/kaggle/input/` es donde Kaggle monta los datasets que agregaste con *Add Input*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Instalar openpyxl para leer archivos .xlsx
!pip install -q openpyxl

# AJUSTAR: ruta dentro de tu Drive donde subiste el Excel.
RUTA_DATASET = "/content/drive/MyDrive/dataset_completo_v2.xlsx"

df = pd.read_excel(RUTA_DATASET, sheet_name="Pares_aumentados")
print(f"{len(df)} filas cargadas de Pares_aumentados.")
print("Columnas:", list(df.columns))
df.head(3)

## 3. Unir con el texto real de cada artículo y armar los pares

El dataset de TSJ trae el NÚMERO de artículo, no su texto — la celda de
abajo lo busca en `articulos_codigo_penal.json` (el export de tu catálogo
de Django) y arma los pares **(hecho, texto del artículo)**.

Con `MultipleNegativesRankingLoss` no hace falta armar negativos a mano:
el resto de los artículos del mismo lote actúan como negativos
automáticamente ("in-batch negatives"). La columna `negativos_dificiles`
que trae `Pares_aumentados` está pensada para negativos EXPLÍCITOS
(tripletas ancla/positivo/negativo, con `TripletLoss`) — si querés
usarla, decime y lo adaptamos; por ahora esta celda la ignora.

In [ ]:
import json
import unicodedata

def _normalizar(texto):
    texto = str(texto).strip().lower()
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))

RUTA_ARTICULOS = "/content/drive/MyDrive/articulos_codigo_penal_v2.json"

with open(RUTA_ARTICULOS, encoding="utf-8") as f:
    articulos_json = json.load(f)

# Mapa (número normalizado, SIGLA normalizada) -> texto del artículo.
# Se indexa por sigla (no por nombre completo) porque distintas normas
# pueden compartir palabras en el nombre ("Código Penal Boliviano" vs.
# "Codigo penal y procedimiento penal" comparten "codigo penal" y las
# dos tienen artículo 335) — la sigla es lo único que no es ambiguo.
mapa_articulos = {}
for a in articulos_json:
    if a["norma_sigla"]:
        mapa_articulos[(_normalizar(a["numero_articulo"]), _normalizar(a["norma_sigla"]))] = a["contenido"]

# AJUSTAR si tu columna "norma" trae otros valores además de "Codigo Penal"
# (corré antes `print(df["norma"].unique())` para ver todos).
# Mapea el nombre TAL COMO VIENE EN EL DATASET DE TSJ -> sigla real del catálogo.
ALIAS_NORMA_A_SIGLA = {
    "codigo penal": "cp",  # -> Código Penal Boliviano, NO Codigo penal y procedimiento penal (CPP)
}

def buscar_texto_articulo(numero, norma_dataset):
    numero_n = _normalizar(numero)
    sigla = ALIAS_NORMA_A_SIGLA.get(_normalizar(norma_dataset))
    if sigla:
        return mapa_articulos.get((numero_n, sigla))
    # Si el valor de "norma" no está en el alias, se intenta como si ya
    # fuera una sigla tal cual (por si el dataset alguna vez trae "CP" directo).
    return mapa_articulos.get((numero_n, _normalizar(norma_dataset)))

df["articulo_texto"] = df.apply(lambda r: buscar_texto_articulo(r["articulo"], r["norma"]), axis=1)

sin_match = df["articulo_texto"].isna().sum()
print(f"Filas SIN texto de artículo encontrado: {sin_match} de {len(df)} ({sin_match/len(df):.1%})")
if sin_match > 0:
    print("\nValores de 'norma' en el dataset que no tienen alias configurado (agregalos a ALIAS_NORMA_A_SIGLA):")
    print(df[df["articulo_texto"].isna()]["norma"].value_counts())
    print("\nEjemplos de (número, norma) sin match:")
    print(df[df["articulo_texto"].isna()][["articulo", "norma"]].drop_duplicates().head(15).to_string(index=False))

df = df.dropna(subset=["articulo_texto"]).reset_index(drop=True)
print(f"\n{len(df)} filas con texto de artículo — listas para entrenar.")

from sentence_transformers import InputExample

ejemplos = [
    InputExample(texts=[str(row["hechos"]), str(row["articulo_texto"])])
    for _, row in df.iterrows()
    if pd.notna(row["hechos"]) and pd.notna(row["articulo_texto"])
]
print(f"{len(ejemplos)} pares de entrenamiento.")

In [ ]:
print(len(df))
print(df["tipo"].value_counts() if "tipo" in df.columns else "no tiene columna 'tipo' -> es Pares_originales")

## 4. Separar un set de validación

Aparta una porción ANTES de entrenar, para poder medir de verdad si el
modelo nuevo mejora — no evalúes con datos que ya vio en el entrenamiento.

In [ ]:
import random
random.seed(42)

# Se separa por fuente_id (la resolución de origen), no por fila al azar:
# una fila "original" y su versión "parafraseado" comparten fuente_id, y si
# terminan una en entrenamiento y otra en validación, el modelo ya "vio"
# ese hecho con otras palabras — el accuracy@3 saldría inflado sin que el
# modelo haya generalizado de verdad.
fuentes = df["fuente_id"].dropna().unique().tolist()
random.shuffle(fuentes)

corte = int(len(fuentes) * 0.9)
fuentes_train = set(fuentes[:corte])
fuentes_val = set(fuentes[corte:])

df_train = df[df["fuente_id"].isin(fuentes_train)]
df_val = df[df["fuente_id"].isin(fuentes_val)]

ejemplos_train = [
    InputExample(texts=[str(row["hechos"]), str(row["articulo_texto"])])
    for _, row in df_train.iterrows()
]
ejemplos_val = [
    InputExample(texts=[str(row["hechos"]), str(row["articulo_texto"])])
    for _, row in df_val.iterrows()
]

print(f"Resoluciones — entrenamiento: {len(fuentes_train)} — validación: {len(fuentes_val)}")
print(f"Pares — entrenamiento: {len(ejemplos_train)} — validación: {len(ejemplos_val)}")

## 5. Cargar el modelo BASE (versión "pre-entrenamiento")

Este es el modelo tal cual, sin afinar — el mismo que hoy corre en
producción (`SENTENCE_TRANSFORMER_MODEL` en `config/settings.py` de
swDerecho). Lo guardamos también, para que el swDerecho pueda tener las
dos versiones (base y afinada) conviviendo — ver el versionado de
embeddings del lado del backend.

In [ ]:
"""
from google.colab import drive

drive.mount("/content/drive")
from sentence_transformers import SentenceTransformer

RUTA_MODELO_BASE = "/content/drive/MyDrive/modelo_base"

modelo = SentenceTransformer(RUTA_MODELO_BASE)

print("Modelo cargado:", RUTA_MODELO_BASE)
"""

## 6. Evaluación ANTES de entrenar (baseline)

`EmbeddingSimilarityEvaluator` no sirve acá porque no tenemos un score de
similitud numérico por par (0 a 1), solo pares positivos. Usamos en cambio
una métrica simple y directa: de los textos de artículo del set de
validación (más un lote de "distractores" del propio dataset), ¿el
artículo correcto queda entre los más cercanos al hecho? Es una forma
casera de accuracy@k, pensada justo para poder comparar antes/después.

In [ ]:
import numpy as np

def evaluar_top_k(modelo_a_evaluar, ejemplos_eval, k=3):
    hechos = [e.texts[0] for e in ejemplos_eval]
    articulos_correctos = [e.texts[1] for e in ejemplos_eval]
    universo_articulos = list(dict.fromkeys(articulos_correctos))  # sin duplicados, orden estable

    emb_hechos = modelo_a_evaluar.encode(hechos, normalize_embeddings=True, show_progress_bar=False)
    emb_universo = modelo_a_evaluar.encode(universo_articulos, normalize_embeddings=True, show_progress_bar=False)

    aciertos = 0
    for i, hecho_emb in enumerate(emb_hechos):
        similitudes = emb_universo @ hecho_emb
        top_k_idx = np.argsort(-similitudes)[:k]
        top_k_articulos = [universo_articulos[j] for j in top_k_idx]
        if articulos_correctos[i] in top_k_articulos:
            aciertos += 1
    return aciertos / len(ejemplos_eval)
"""
acc_base = evaluar_top_k(modelo, ejemplos_val, k=3)
print(f"Accuracy@3 del modelo BASE (sin fine-tuning): {acc_base:.1%}")

# Cargar y evaluar el modelo ya afinado (v1)
modelo_v1 = SentenceTransformer("/content/drive/MyDrive/sw-derecho-embeddings-v1-final")
acc_v1 = evaluar_top_k(modelo_v1, ejemplos_val, k=3)
print(f"Accuracy@3 del modelo v1 (afinado, corrida anterior): {acc_v1:.1%}")
"""

In [ ]:
#print("Artículos únicos en el universo de validación:", len(set(str(e.texts[1]) for e in ejemplos_val)))

## 7. Entrenar

In [ ]:
import os
import json
import torch

from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses

# ============================================================
# CONFIGURACIÓN — MODELO B
# ============================================================

BATCH_SIZE = 32
EPOCHS = 4
MAX_SEQ_LENGTH = 256

# Carpeta EXCLUSIVA para Modelo B
DRIVE_PATH = "/content/drive/MyDrive/sw-derecho-embeddings-modelo-B-224"

PROGRESS_FILE = os.path.join(
    DRIVE_PATH,
    "training_progress.json"
)

os.makedirs(DRIVE_PATH, exist_ok=True)

print("============================================")
print("MODELO B — LONGITUD CORREGIDA")
print("============================================")
print(f"Batch size: {BATCH_SIZE}")
print(f"Épocas: {EPOCHS}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Carpeta: {DRIVE_PATH}")
print()


# ============================================================
# BUSCAR CHECKPOINT
# ============================================================

def buscar_ultimo_checkpoint():

    checkpoints = []

    for nombre in os.listdir(DRIVE_PATH):

        if nombre.startswith("checkpoint-epoch-"):

            try:
                numero = int(
                    nombre.replace(
                        "checkpoint-epoch-",
                        ""
                    )
                )

                ruta = os.path.join(
                    DRIVE_PATH,
                    nombre
                )

                if os.path.isdir(ruta):
                    checkpoints.append(
                        (numero, ruta)
                    )

            except ValueError:
                pass

    if not checkpoints:
        return None, 0

    checkpoints.sort(
        key=lambda x: x[0]
    )

    numero, ruta = checkpoints[-1]
    return ruta, numero


checkpoint_resultado = buscar_ultimo_checkpoint()

if checkpoint_resultado[0] is not None:

    checkpoint_path, ultima_epoca = checkpoint_resultado

else:

    checkpoint_path = None
    ultima_epoca = 0


# ============================================================
# CARGAR MODELO
# ============================================================

if checkpoint_path is not None:

    print("============================================")
    print("CHECKPOINT DE MODELO B ENCONTRADO")
    print("============================================")

    print(f"Última época: {ultima_epoca}")
    print(f"Checkpoint: {checkpoint_path}")
    print()

    modelo = SentenceTransformer(
        checkpoint_path
    )

else:

    print("============================================")
    print("CARGANDO MODELO BASE")
    print("============================================")

    print(
        "Descargando/cargando "
        "paraphrase-multilingual-mpnet-base-v2..."
    )

    modelo = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    )

    print("Modelo base cargado.")


# ============================================================
# CONFIGURAR LONGITUD
# ============================================================

modelo.max_seq_length = MAX_SEQ_LENGTH


# ============================================================
# VERIFICACIÓN
# ============================================================

print()
print("============================================")
print("CONFIGURACIÓN DEL MODELO")
print("============================================")

print(
    "max_seq_length:",
    modelo.max_seq_length
)

print(
    "max_position_embeddings:",
    modelo[0].auto_model.config.max_position_embeddings
)

print()


# ============================================================
# DATALOADER
# ============================================================

train_dataloader = DataLoader(
    ejemplos_train,
    shuffle=True,
    batch_size=BATCH_SIZE
)

print(
    "Ejemplos de entrenamiento:",
    len(ejemplos_train)
)

print(
    "Batches por época:",
    len(train_dataloader)
)

print()


# ============================================================
# SI YA TERMINÓ
# ============================================================

if ultima_epoca >= EPOCHS:

    print("============================================")
    print("ENTRENAMIENTO YA COMPLETADO")
    print("============================================")

    print(
        f"Épocas completadas: "
        f"{ultima_epoca}/{EPOCHS}"
    )

    print(
        "Modelo disponible en:"
    )

    print(DRIVE_PATH)


# ============================================================
# ENTRENAMIENTO
# ============================================================

else:

    for epoca in range(
        ultima_epoca + 1,
        EPOCHS + 1
    ):

        print()
        print("============================================")
        print(
            f"INICIANDO ÉPOCA "
            f"{epoca}/{EPOCHS}"
        )
        print("============================================")

        # --------------------------------------------
        # LOSS
        # --------------------------------------------

        train_loss = losses.MultipleNegativesRankingLoss(
            modelo
        )

        # --------------------------------------------
        # WARMUP
        # --------------------------------------------

        warmup_steps = int(
            len(train_dataloader) * 0.1
        )

        # --------------------------------------------
        # CHECKPOINT
        # --------------------------------------------

        checkpoint_actual = os.path.join(
            DRIVE_PATH,
            f"checkpoint-epoch-{epoca}"
        )

        print(
            "Guardando checkpoint en:"
        )

        print(checkpoint_actual)

        # --------------------------------------------
        # ENTRENAR 1 ÉPOCA
        # --------------------------------------------

        modelo.fit(
            train_objectives=[
                (
                    train_dataloader,
                    train_loss
                )
            ],
            epochs=1,
            warmup_steps=warmup_steps,
            output_path=checkpoint_actual,
            show_progress_bar=True
        )

        # --------------------------------------------
        # GUARDAR PROGRESO
        # --------------------------------------------

        progreso = {
            "modelo": "B",
            "base_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            "max_seq_length": MAX_SEQ_LENGTH,
            "ultima_epoca_completada": epoca,
            "total_epocas": EPOCHS,
            "batch_size": BATCH_SIZE,
            "checkpoint": checkpoint_actual
        }

        with open(
            PROGRESS_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                progreso,
                f,
                indent=4,
                ensure_ascii=False
            )

        print()
        print("ÉPOCA COMPLETADA")
        print(
            f"Época: {epoca}/{EPOCHS}"
        )

        print(
            "Checkpoint guardado:"
        )

        print(checkpoint_actual)

        # Asegurar longitud
        modelo.max_seq_length = MAX_SEQ_LENGTH


# ============================================================
# FIN
# ============================================================

print()
print("============================================")
print("MODELO B FINALIZADO")
print("============================================")

print(
    f"Épocas: {EPOCHS}/{EPOCHS}"
)

print(
    "max_seq_length:",
    modelo.max_seq_length
)

print(
    "Modelo/checkpoints:"
)

print(DRIVE_PATH)

## 8. Evaluación DESPUÉS de entrenar — comparación directa

In [ ]:
acc_finetuned = evaluar_top_k(modelo, ejemplos_val, k=3)

print(f"Accuracy@3 — modelo AFINADO:   {acc_finetuned:.1%}")

In [ ]:
from collections import Counter
frecuencias = Counter(str(e.texts[1]) for e in ejemplos_val)
top3 = sum(c for _, c in frecuencias.most_common(3))
print(f"Baseline 'siempre los 3 artículos más frecuentes': {top3/len(ejemplos_val):.1%}")
print("Artículos únicos en el universo de validación:", len(set(str(e.texts[1]) for e in ejemplos_val)))

In [ ]:
from sklearn.metrics import classification_report
from collections import Counter
import numpy as np

def evaluar_por_articulo(modelo_a_evaluar, df_eval):
    """Igual idea que antes, pero usa el NÚMERO de artículo (columna
    'articulo' de tu DataFrame) como etiqueta — no el texto completo —
    para que el reporte sea legible."""
    hechos = df_eval["hechos"].tolist()
    articulos_correctos = df_eval["articulo"].astype(str).tolist()

    mapa_num_a_texto = dict(zip(df_eval["articulo"].astype(str), df_eval["articulo_texto"]))
    universo_numeros = list(mapa_num_a_texto.keys())
    universo_textos = [mapa_num_a_texto[n] for n in universo_numeros]

    emb_hechos = modelo_a_evaluar.encode(hechos, normalize_embeddings=True, show_progress_bar=False)
    emb_universo = modelo_a_evaluar.encode(universo_textos, normalize_embeddings=True, show_progress_bar=False)

    predicciones = []
    for hecho_emb in emb_hechos:
        similitudes = emb_universo @ hecho_emb
        idx_mejor = np.argmax(similitudes)
        predicciones.append(universo_numeros[idx_mejor])

    return articulos_correctos, predicciones


y_true, y_pred = evaluar_por_articulo(modelo, df_val)

# Solo los 25 artículos con más ejemplos en validación — el reporte
# completo con 203 artículos sigue siendo largo, esto lo hace manejable.
top_articulos = [a for a, _ in Counter(y_true).most_common(25)]
mascara = [t in top_articulos for t in y_true]
y_true_top = [t for t, m in zip(y_true, mascara) if m]
y_pred_top = [p for p, m in zip(y_pred, mascara) if m]

print(classification_report(y_true_top, y_pred_top, zero_division=0))

In [ ]:
def evaluar_ranking_completo(modelo_a_evaluar, ejemplos_eval, ks=(1, 3, 5)):
    hechos = [e.texts[0] for e in ejemplos_eval]
    articulos_correctos = [str(e.texts[1]) for e in ejemplos_eval]
    universo_articulos = list(dict.fromkeys(articulos_correctos))

    emb_hechos = modelo_a_evaluar.encode(hechos, normalize_embeddings=True, show_progress_bar=False)
    emb_universo = modelo_a_evaluar.encode(universo_articulos, normalize_embeddings=True, show_progress_bar=False)

    reciprocal_ranks = []
    aciertos_en_k = {k: 0 for k in ks}

    for i, hecho_emb in enumerate(emb_hechos):
        similitudes = emb_universo @ hecho_emb
        orden = np.argsort(-similitudes)
        articulos_ordenados = [universo_articulos[j] for j in orden]

        # Posición (1-indexada) del artículo correcto en el ranking completo.
        posicion = articulos_ordenados.index(articulos_correctos[i]) + 1
        reciprocal_ranks.append(1.0 / posicion)

        for k in ks:
            if articulos_correctos[i] in articulos_ordenados[:k]:
                aciertos_en_k[k] += 1

    n = len(ejemplos_eval)
    mrr = sum(reciprocal_ranks) / n

    print(f"MRR (posición promedio invertida del artículo correcto): {mrr:.3f}")
    print(f"(un MRR de 1.0 sería 'siempre en el puesto 1'; 0.33 sería 'en promedio, puesto 3')\n")

    for k in ks:
        recall_k = aciertos_en_k[k] / n          # con 1 solo artículo correcto por hecho, recall@k = accuracy@k
        precision_k = recall_k / k                # de los k que mostrás, cuántos son relevantes en promedio
        f1_k = 2 * precision_k * recall_k / (precision_k + recall_k) if (precision_k + recall_k) > 0 else 0
        print(f"k={k} -> Precision@{k}: {precision_k:.1%}  Recall@{k}: {recall_k:.1%}  F1@{k}: {f1_k:.1%}")

    return mrr, aciertos_en_k


mrr, aciertos = evaluar_ranking_completo(modelo, ejemplos_val)

In [ ]:
print(classification_report(y_true_top, y_pred_top, labels=top_articulos, zero_division=0))

In [ ]:
from collections import Counter
errores = Counter((t, p) for t, p in zip(y_true, y_pred) if t != p)
for articulo in ['13', '154', '283']:
    print(f"\nConfusiones cuando el correcto era {articulo}:")
    for (correcto, predicho), cantidad in errores.most_common():
        if correcto == articulo:
            print(f"  -> predijo {predicho}: {cantidad} veces")

In [ ]:
tok = modelo.tokenizer
lens = np.array([len(tok(h)["input_ids"]) for h in df["hechos"].sample(500, random_state=1)])
print(np.percentile(lens, [50, 90]), (lens > 128).mean())
import numpy as np
tok = modelo.tokenizer

# 1) ¿Qué hay en la parte cortada? (un ejemplo)
ids = tok(df["hechos"].iloc[0])["input_ids"]
print("Se ve:  ", tok.decode(ids[:128]))
print("Se corta:", tok.decode(ids[128:]))

# 2) Largo de los artículos
art = df["articulo_texto"].drop_duplicates()
la = np.array([len(tok(t)["input_ids"]) for t in art])
print("artículos >128:", (la > 128).mean(), " p90:", np.percentile(la, 90), " max:", la.max())
print("hechos max:", lens.max())

In [ ]:
import numpy as np

tok = modelo.tokenizer

tokens_hechos = np.array([
    len(tok(
        str(x),
        add_special_tokens=True,
        truncation=False
    )["input_ids"])
    for x in df["hechos"]
])

print("============================================")
print("DISTRIBUCIÓN REAL DE TOKENS — HECHOS")
print("============================================")

print("P50 :", np.percentile(tokens_hechos, 50))
print("P90 :", np.percentile(tokens_hechos, 90))
print("P95 :", np.percentile(tokens_hechos, 95))
print("P99 :", np.percentile(tokens_hechos, 99))
print("MAX :", tokens_hechos.max())

for limite in [128, 160, 192, 224, 256, 320, 384, 448, 512]:

    cantidad = (tokens_hechos > limite).sum()
    porcentaje = cantidad / len(tokens_hechos) * 100

    print(
        f">{limite}: "
        f"{cantidad:,} "
        f"({porcentaje:.2f}%)"
    )
tokens_articulos = np.array([
    len(tok(
        str(x),
        add_special_tokens=True,
        truncation=False
    )["input_ids"])
    for x in df["articulo_texto"].drop_duplicates()
])

print("============================================")
print("DISTRIBUCIÓN REAL DE TOKENS — ARTÍCULOS")
print("============================================")

print("P50 :", np.percentile(tokens_articulos, 50))
print("P90 :", np.percentile(tokens_articulos, 90))
print("P95 :", np.percentile(tokens_articulos, 95))
print("P99 :", np.percentile(tokens_articulos, 99))
print("MAX :", tokens_articulos.max())

for limite in [128, 160, 192, 224, 256, 320, 384, 448, 512]:

    cantidad = (tokens_articulos > limite).sum()
    porcentaje = cantidad / len(tokens_articulos) * 100

    print(
        f">{limite}: "
        f"{cantidad:,} "
        f"({porcentaje:.2f}%)"
    )

## 9. Empaquetar para descargar

Kaggle guarda todo lo que quede en `/kaggle/working/` como *Output* del
notebook cuando hacés *Save Version*. Este zip es solo para bajarlo más
fácil en un solo archivo.

## 9. Guardar en Drive (¡antes de que se corte la sesión!)

A diferencia de Kaggle, acá el almacenamiento local (`/content/`) se
borra apenas termina la sesión — hay que copiar el modelo a Drive
explícitamente antes de cerrar la pestaña o que se desconecte por
inactividad (90 min sin uso).

In [ ]:
modelo.save("/content/drive/MyDrive/sw-derecho-embeddings-v4-final")
import os
print(os.listdir("/content/drive/MyDrive/sw-derecho-embeddings-v4-final"))

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_guardado = SentenceTransformer("/content/drive/MyDrive/sw-derecho-embeddings-v4-final")
print(f"Accuracy@3 del modelo guardado: {evaluar_top_k(modelo_guardado, ejemplos_val, k=3):.1%}")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/sw-derecho-embeddings-v4-final"))

In [ ]:
# from collections import Counter
# frecuencias = Counter(str(e.texts[1]) for e in ejemplos_val)
# top3 = sum(c for _, c in frecuencias.most_common(3))
# print(f"Baseline 'siempre los 3 artículos más frecuentes': {top3/len(ejemplos_val):.1%}")

In [ ]:
modelo.save("/content/drive/MyDrive/sw-derecho-embeddings-v4-final")